# 09 — Phase & Rotation Gates

Pick up the dials that send a qubit to *any* point on the Bloch sphere: the phase gates $S, S^\dagger$ and the continuous rotations $R_x, R_y, R_z$.

**Prerequisites:** `01/07_pauli_gates`, `01/08_hadamard_and_measurement_bases`.

**What you'll be able to do**
- Use the phase gate $S = \mathrm{diag}(1, i)$ and its inverse $S^\dagger$ (the `sdg` from `01/08`), and see $S^2 = Z$.
- Spin a state's longitude with $P(\lambda) = \mathrm{diag}(1, e^{i\lambda})$, and tilt its latitude with $R_x, R_y, R_z$.
- Reach an arbitrary single-qubit state by composing rotations.

The Paulis are half-turns — coarse. To prepare a qubit at a *chosen* latitude and longitude we need fine, continuous control. These are those knobs, and they close the single-qubit toolkit: with them you can build any state and any single-qubit gate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_PLUS, bloch_vector
from qot_course.quantum.gates import (
    phase_gate, rx, ry, rz, S, S_DAG, PAULI_Z, IDENTITY, apply_gate, is_unitary,
)

np.random.seed(42)
viz.use_course_style()

## 1. The phase gate S and its inverse

$$ S = \begin{pmatrix}1&0\\0&i\end{pmatrix} = P(\pi/2), \qquad S^\dagger = \begin{pmatrix}1&0\\0&-i\end{pmatrix}. $$

$S$ adds a quarter-turn of phase to the $|1\rangle$ amplitude. Applying it twice gives $S^2 = Z$ (two quarter-turns make the half-turn phase flip), and its inverse $S^\dagger$ is the `sdg` gate that `01/08` used to reach the $Y$ basis.

In [ ]:
print("S =\n", np.round(S, 3))
print("S^2 = Z ?", np.allclose(S @ S, PAULI_Z))
print("S . S^dagger = I ?", np.allclose(S @ S_DAG, IDENTITY))
print("S|+> =", np.round(apply_gate(S, KET_PLUS), 3), " (= |+i>: + rotated a quarter-turn on the equator)")

**Read the output.** $S$ squares to the phase flip $Z$, $S^\dagger$ undoes $S$, and acting on $|+\rangle$ it produces $|+i\rangle$ — the same equatorial state we measured in `01/08`. $S$ is a $90^\circ$ spin of the longitude; $Z$ is the $180^\circ$ version.

## 2. The general phase gate spins the longitude

$$ P(\lambda) = \begin{pmatrix}1&0\\0&e^{i\lambda}\end{pmatrix}. $$

$P(\lambda)$ leaves the weights of $|0\rangle$ and $|1\rangle$ alone — it only changes the *relative phase* from `01/02`. On the Bloch sphere that is a rotation around the equator: it moves longitude, not latitude. Watch $|+\rangle$ trace the equator as $\lambda$ runs from $0$ to $2\pi$.

In [ ]:
lambdas = np.linspace(0, 2 * np.pi, 200)
xs = [bloch_vector(apply_gate(phase_gate(lam), KET_PLUS))[0] for lam in lambdas]
ys = [bloch_vector(apply_gate(phase_gate(lam), KET_PLUS))[1] for lam in lambdas]

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.plot(xs, ys, color=COLORS["quantum"], lw=2.5)
ax.scatter([1], [0], color=COLORS["highlight"], zorder=5, s=60)
ax.annotate(r"$|+\rangle$ at $\lambda=0$", (1, 0), textcoords="offset points", xytext=(-30, 10), color=COLORS["text"])
ax.axhline(0, color=COLORS["grid"], lw=1); ax.axvline(0, color=COLORS["grid"], lw=1)
ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
ax.set_xlabel(r"$\langle X\rangle$"); ax.set_ylabel(r"$\langle Y\rangle$")
ax.set_title(r"$P(\lambda)$ spins $|+\rangle$ around the equator", pad=12)
plt.show()

**Read the figure.** As $\lambda$ grows, $|+\rangle$ slides around the equator of the Bloch sphere (seen from above, the $\langle X\rangle$-$\langle Y\rangle$ plane) without ever leaving it — a pure change of longitude. This is the continuous version of the discrete phase steps $S$ and $Z$, and the direct handle on the relative phase that interference cares about.

## 3. The rotation gates tilt the latitude

The Pauli rotations give continuous turns about each axis:

$$ R_a(\theta) = e^{-i\theta\,\sigma_a/2} = \cos(\theta/2)\,I - i\sin(\theta/2)\,\sigma_a, \qquad a \in \{x, y, z\}. $$

$R_z(\theta)$ spins the longitude (like $P$, up to a global phase); $R_x$ and $R_y$ *tilt the latitude*. Sweeping $R_y(\theta)$ takes $|0\rangle$ down a meridian from the north pole to the south pole.

In [ ]:
for g, name in [(rx(0.7), "Rx(0.7)"), (ry(0.7), "Ry(0.7)"), (rz(0.7), "Rz(0.7)")]:
    print(f"{name}: unitary = {is_unitary(g)}")

thetas = np.linspace(0, np.pi, 200)
coords = np.array([bloch_vector(apply_gate(ry(t), KET_0)) for t in thetas])

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.plot(coords[:, 0], coords[:, 2], color=COLORS["flow"], lw=2.5)
ax.scatter([0, 0], [1, -1], color=COLORS["highlight"], zorder=5, s=60)
ax.annotate(r"$|0\rangle$", (0, 1), textcoords="offset points", xytext=(8, 0), color=COLORS["text"])
ax.annotate(r"$|1\rangle$", (0, -1), textcoords="offset points", xytext=(8, 0), color=COLORS["text"])
ax.axhline(0, color=COLORS["grid"], lw=1); ax.axvline(0, color=COLORS["grid"], lw=1)
ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
ax.set_xlabel(r"$\langle X\rangle$"); ax.set_ylabel(r"$\langle Z\rangle$")
ax.set_title(r"$R_y(\theta)$ tilts $|0\rangle$ down a meridian", pad=12)
plt.show()

**Read the figure.** $R_y(\theta)$ carries $|0\rangle$ along a meridian (the $\langle X\rangle$-$\langle Z\rangle$ plane), reaching the equator at $\theta=\pi/2$ and the south pole $|1\rangle$ at $\theta=\pi$. Latitude is now a continuous dial. Combine a latitude tilt ($R_y$) with a longitude spin ($R_z$ or $P$) and you can place a qubit at *any* point on the sphere.

## 4. Any single-qubit gate is a product of rotations

A foundational fact closes the toolkit: every single-qubit unitary can be written as a sequence of rotations, $U = e^{i\alpha} R_z(\beta)\,R_y(\gamma)\,R_z(\delta)$ (an overall phase times three turns). We will not prove it here, but it is why $R_x, R_y, R_z$ — plus the phase gates — are *complete* for one qubit: nothing single-qubit lies beyond them. (Nielsen & Chuang, Theorem 4.1.)

## Your turn

1. **Quarter-turns.** Confirm $S^2 = Z$ and $S^4 = I$. How many applications of $S$ return you to the start, and why?
2. **Aim a probability.** Find the $\theta$ for which $R_y(\theta)|0\rangle$ gives $P(0) = 0.25$. (Hint: $P(0) = \cos^2(\theta/2)$ — the same closed form as `01/04`.)
3. **Phase is invisible to Z.** Apply $R_z(\theta)$ to $|+\rangle$ for a few $\theta$ and confirm a $Z$-basis measurement never changes — then confirm an $X$-basis measurement (via `01/08`) does. Where did the information go?

## What you built

- You used $S = P(\pi/2)$ and $S^\dagger$ (the `sdg` of `01/08`), and saw $S^2 = Z$.
- You spun a state's longitude with the phase gate $P(\lambda)$ and tilted its latitude with $R_x, R_y, R_z$.
- You learned that composing rotations reaches any single-qubit state and any single-qubit gate.

You now hold the full set of single-qubit controls — coarse half-turns and fine continuous dials alike. That is everything needed to drive one qubit anywhere on its sphere.

## What's next

We keep reading $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ off states. In `01/10_expectation_values` we make the expectation value $\langle A\rangle = \langle\psi|A|\psi\rangle$ precise, estimate it from finite samples, and connect it straight to the tomography of `01/15`.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 4.2 (single-qubit gates), Theorem 4.1, Cambridge University Press (2010).
- Qiskit textbook, "More Circuit Identities" (rotation gates).

**Previous:** `notebooks/01_QuantumFoundations/08_hadamard_and_measurement_bases.ipynb` · **Next:** `notebooks/01_QuantumFoundations/10_expectation_values.ipynb`